In [ ]:
import duckdb 
from pathlib import Path
import csv

con = duckdb.connect('../data/capstone_data.duckdb')

raw_path = Path("../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv")  
new_clean_path = Path("../data/processed/transactions_clean_v2.csv")

In [18]:
columns = [
    "ATM_Address", "ATM_City_State", "Merchant_Category", "Merchant_Name",
    "Transaction_Code", "Transaction_Code_Description", "Transaction_Type",
    "Transaction_Type_Description", "Transaction_Number", "Amount_Completed",
    "Transaction_Date", "Time_Local_hhmmss", "Recurring_Trxn", "CustomerID"
]

Getting columns names from the existing table and stroing them all as 'varchar' for now to avoid any type errors

In [2]:
column_names = con.execute("DESCRIBE transactions").df()['column_name'].tolist()
print(f"Column names: {column_names}")

columns_dict = {col: 'VARCHAR' for col in column_names}
print(f"Columns: {columns_dict}")

Column names: ['ATM_Address', 'ATM_City_State', 'Merchant_Category', 'Merchant_Name', 'Transaction_Code', 'Transaction_Code_Description', 'Transaction_Type', 'Transaction_Type_Description', 'Transaction_Number', 'Amount_Completed', 'Transaction_Date', 'Time_Local_hhmmss', 'Recurring_Trxn', 'CustomerID']
Columns: {'ATM_Address': 'VARCHAR', 'ATM_City_State': 'VARCHAR', 'Merchant_Category': 'VARCHAR', 'Merchant_Name': 'VARCHAR', 'Transaction_Code': 'VARCHAR', 'Transaction_Code_Description': 'VARCHAR', 'Transaction_Type': 'VARCHAR', 'Transaction_Type_Description': 'VARCHAR', 'Transaction_Number': 'VARCHAR', 'Amount_Completed': 'VARCHAR', 'Transaction_Date': 'VARCHAR', 'Time_Local_hhmmss': 'VARCHAR', 'Recurring_Trxn': 'VARCHAR', 'CustomerID': 'VARCHAR'}


In [3]:
# Letting DuckDB auto-detect everything first
raw_sample = con.execute("""
    SELECT * FROM read_csv_auto(
        '../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv',
        sample_size=1000
    )
    LIMIT 5
""").df()

print("Raw file columns:")
print(raw_sample.columns.tolist())
print(f"\nRaw file has {len(raw_sample.columns)} columns")

print("\nProcessed table columns:")
print(column_names)
print(f"Processed table has {len(column_names)} columns")

Raw file columns:
['609653 BROADWAY,BANGORME,5310,TJMAXX #0433,20,POS Debit        Pri,P,Pinned Settlements  ,25,32.99,2025-11-23,141633, ,BSB149015']

Raw file has 1 columns

Processed table columns:
['ATM_Address', 'ATM_City_State', 'Merchant_Category', 'Merchant_Name', 'Transaction_Code', 'Transaction_Code_Description', 'Transaction_Type', 'Transaction_Type_Description', 'Transaction_Number', 'Amount_Completed', 'Transaction_Date', 'Time_Local_hhmmss', 'Recurring_Trxn', 'CustomerID']
Processed table has 14 columns


This is the query which has been detected as giving the errors since the cleaner python function has put all the columns from the left side in the merchant column e.g - Merchant_Name : Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE

In [4]:
# Get specific transactions with NULL ATM_City_State
null_transactions = con.execute("""
    SELECT 
        CustomerID,
        Transaction_Date,
        Time_Local_hhmmss,
        Amount_Completed,
        Merchant_Name,
        ATM_City_State
    FROM transactions 
    WHERE ATM_City_State IS NULL
""").df()

print(f" {len(null_transactions)} Transactions with NULL ATM_City_State:")
print(null_transactions.T)

 928 Transactions with NULL ATM_City_State:
                                                           0    \
CustomerID                                           BSB066832   
Transaction_Date                           2025-11-23 00:00:00   
Time_Local_hhmmss                                        91146   
Amount_Completed                                          49.0   
Merchant_Name      Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE   
ATM_City_State                                            None   

                                                            1    \
CustomerID                                            BSB070691   
Transaction_Date                            2025-11-22 00:00:00   
Time_Local_hhmmss                                         65052   
Amount_Completed                                         107.94   
Merchant_Name       701 Tillery S,AUSTINTX,5712,WWW.AETICON.COM   
ATM_City_State                                             None   

                       

Checking the data in the original raw csv file

In [5]:
# Search for this specific transaction (first from the previous query)
target_customer = 'BSB066832'
target_date = '2025-11-23'
target_amount = '49.0'  

print(f"Searching for: Customer {target_customer}, Date {target_date}, Amount {target_amount}")

with open('../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv', 'r') as f:
    for line_num, line in enumerate(f, 1):
        if target_customer in line and target_date in line:
            # Check for amount variations
            if '49.0' in line or ',49,' in line or ',49.00,' in line:
                print(f"\nFound at line {line_num}:")
                print(line.strip())
                print("\n--- Field-by-field split (first 15 fields) ---")
                parts = line.split(',')
                for i, part in enumerate(parts[:15]):
                    print(f"Field {i:2d}: '{part}'")
                break

Searching for: Customer BSB066832, Date 2025-11-23, Amount 49.0

Found at line 2946026:
7124,Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE,40,D/C SETTLEMENT,X,Pinless Settlements ,24,49.00,2025-11-23,91146, ,BSB066832

--- Field-by-field split (first 15 fields) ---
Field  0: '7124'
Field  1: 'Covered Bridge'
Field  2: 'AUSTINTX'
Field  3: '5815'
Field  4: 'TRUST ME MOVIE'
Field  5: '40'
Field  6: 'D/C SETTLEMENT'
Field  7: 'X'
Field  8: 'Pinless Settlements '
Field  9: '24'
Field 10: '49.00'
Field 11: '2025-11-23'
Field 12: '91146'
Field 13: ' '
Field 14: 'BSB066832
'


As expected the above query has shown 15 columns as expected to the actual 14 columns

Now, checking the data in the transactions table which should have already been clean but looks like the python parser failed on certain transactions

In [6]:
# Get more examples with different patterns
examples = con.execute("""
    SELECT CustomerID, Transaction_Date, Amount_Completed, Merchant_Category, Merchant_Name
    FROM transactions 
    WHERE ATM_City_State IS NULL
    LIMIT 5
""").df()

print(examples)

  CustomerID Transaction_Date  Amount_Completed Merchant_Category  \
0  BSB066832       2025-11-23             49.00              7124   
1  BSB070691       2025-11-22            107.94              1178   
2  BSB104160       2025-11-23            145.44               191   
3  BSB171309       2025-11-22             93.25              2174   
4  BSB171309       2025-11-21             11.70              2174   

                                       Merchant_Name  
0        Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE  
1        701 Tillery S,AUSTINTX,5712,WWW.AETICON.COM  
2   East Main Stre,SearsportME,5949,QLT*Keepsakes LL  
3               ROUTE 161,NANTESQC,5499,VOISIN #2140  
4               ROUTE 161,NANTESQC,5499,VOISIN #2140  


Making sure with checking the transactions in the raw file

In [7]:
# Search for these specific transactions in raw
search_terms = [
    ('BSB066832', '2025-11-23', '49'),
    ('BSB070691', '2025-11-22', '107.94'),
    ('BSB104160', '2025-11-23', '145.44'),
]

with open('../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv', 'r') as f:
    for line_num, line in enumerate(f, 1):
        for cust, date, amt in search_terms:
            if cust in line and date in line:
                print(f"\n{'='*80}")
                print(f"Customer {cust}:")
                print(line.strip())
                print("\nFields:")
                parts = line.split(',')
                for i, part in enumerate(parts):
                    print(f"  [{i:2d}] '{part}'")
                search_terms.remove((cust, date, amt))
                break
        if not search_terms:
            break


Customer BSB104160:
70 SPRINGER DR,BANGORME,5200,LOWE S #1940,20,POS Debit        Pri,P,Pinned Settlements  ,95,31.00,2025-11-23,173318, ,BSB104160

Fields:
  [ 0] '70 SPRINGER DR'
  [ 1] 'BANGORME'
  [ 2] '5200'
  [ 3] 'LOWE S #1940'
  [ 4] '20'
  [ 5] 'POS Debit        Pri'
  [ 6] 'P'
  [ 7] 'Pinned Settlements  '
  [ 8] '95'
  [ 9] '31.00'
  [10] '2025-11-23'
  [11] '173318'
  [12] ' '
  [13] 'BSB104160
'

Customer BSB066832:
7124,Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE,40,D/C SETTLEMENT,X,Pinless Settlements ,24,49.00,2025-11-23,91146, ,BSB066832

Fields:
  [ 0] '7124'
  [ 1] 'Covered Bridge'
  [ 2] 'AUSTINTX'
  [ 3] '5815'
  [ 4] 'TRUST ME MOVIE'
  [ 5] '40'
  [ 6] 'D/C SETTLEMENT'
  [ 7] 'X'
  [ 8] 'Pinless Settlements '
  [ 9] '24'
  [10] '49.00'
  [11] '2025-11-23'
  [12] '91146'
  [13] ' '
  [14] 'BSB066832
'

Customer BSB070691:
1178, 701 Tillery S,AUSTINTX,5712,WWW.AETICON.COM,40,D/C SETTLEMENT,X,Pinless Settlements ,77,107.94,2025-11-22,65052, ,BSB070691

Fields:
  [ 0

Comparing with good data rows where ATM_City_State is not null

In [8]:
# Find transactions WITH ATM_City_State to compare structure
with_city = con.execute("""
    SELECT CustomerID, Transaction_Date, Amount_Completed, 
           ATM_Address, ATM_City_State, Merchant_Category, Merchant_Name
    FROM transactions 
    WHERE ATM_City_State IS NOT NULL
    LIMIT 3
""").df()

print("\nTransactions WITH ATM_City_State:")
print(with_city)


Transactions WITH ATM_City_State:
  CustomerID Transaction_Date  Amount_Completed         ATM_Address  \
0  BSB149015       2025-11-23             32.99     609653 BROADWAY   
1  BSB366107       2025-11-24             91.08  HANNAFORD #8241 93   
2  BSB478058       2025-11-23             57.23             BUFFALO   

  ATM_City_State Merchant_Category    Merchant_Name  
0       BANGORME              5310     TJMAXX #0433  
1      BELFASTME              5411  HANNAFORD #8241  
2      BUFFALONY              5411    DASH S MARKET  


Changing and modifying the original python parser with using MCC code as a fixed index of 2 but fails when the firct column ATM_Address has a comma in it (see Test2 and Test3)

In [9]:
# Test the new parsing on a small sample
def parse_line_v2(line):
    """Updated version with positional MCC detection"""
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts  # clean line
    
    # Right 10 columns are stable
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # MCC should be at index 2 from the left
    if len(left_remainder) >= 3:
        atm_address = left_remainder[0]
        atm_city_state = left_remainder[1] if left_remainder[1].strip() else ''
        merchant_category = left_remainder[2]
        merchant_name = ','.join(left_remainder[3:])  # Join any extra parts
        return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10
    
    return None

# Test on your known problem cases
test_lines = [
    "70 SPRINGER DR,BANGORME,5200,LOWE S #1940,20,POS Debit        Pri,P,Pinned Settlements  ,95,31.00,2025-11-23,173318, ,BSB104160",
    "7124,Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE,40,D/C SETTLEMENT,X,Pinless Settlements ,24,49.00,2025-11-23,91146, ,BSB066832",
    "1178, 701 Tillery S,AUSTINTX,5712,WWW.AETICON.COM,40,D/C SETTLEMENT,X,Pinless Settlements ,77,107.94,2025-11-22,65052, ,BSB070691"
]

print("Testing new parser on known cases:\n")
for i, line in enumerate(test_lines, 1):
    result = parse_line_v2(line)
    if result:
        print(f"Test {i}:")
        print(f"  ATM_Address: {result[0]}")
        print(f"  ATM_City_State: {result[1]}")
        print(f"  MCC: {result[2]}")
        print(f"  Merchant_Name: {result[3]}")
        print(f"  CustomerID: {result[-1]}")
        print()

Testing new parser on known cases:

Test 1:
  ATM_Address: 70 SPRINGER DR
  ATM_City_State: BANGORME
  MCC: 5200
  Merchant_Name: LOWE S #1940
  CustomerID: BSB104160

Test 2:
  ATM_Address: 7124
  ATM_City_State: Covered Bridge
  MCC: AUSTINTX
  Merchant_Name: 5815,TRUST ME MOVIE
  CustomerID: BSB066832

Test 3:
  ATM_Address: 1178
  ATM_City_State:  701 Tillery S
  MCC: AUSTINTX
  Merchant_Name: 5712,WWW.AETICON.COM
  CustomerID: BSB070691



In [10]:
def parse_line_v3(line):
    """Find MCC by valid code pattern, not just any numeric"""
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts  # clean line
    
    # Right 10 columns are stable
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # Find MCC: 3-4 digits starting with 0-7 (valid MCC range)
    mcc_index = None
    for i, val in enumerate(left_remainder):
        if (val.strip().isdigit() and 
            len(val.strip()) in [3, 4] and
            val.strip()[0] in '01234567'):  # MCCs start with 0-7
            mcc_index = i
            break  # Take first valid MCC pattern
    
    if mcc_index is None:
        return None
    
    # Now build fields based on MCC position
    if mcc_index == 2:  # Normal 14-field structure
        atm_address = left_remainder[0]
        atm_city_state = left_remainder[1]
        merchant_category = left_remainder[2]
        merchant_name = ','.join(left_remainder[3:])
    elif mcc_index == 3:  # 15-field structure with merchant location
        atm_address = left_remainder[0]
        atm_city_state = ''  # NULL - no ATM location for card transactions
        merchant_category = left_remainder[3]
        # Combine merchant street/city with name
        merchant_name = ','.join(left_remainder[1:3] + left_remainder[4:])
    else:
        return None  # Unexpected structure
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Test again
print("Testing v3 parser:\n")
for i, line in enumerate(test_lines, 1):
    result = parse_line_v3(line)
    if result:
        print(f"Test {i}:")
        print(f"  ATM_Address: {result[0]}")
        print(f"  ATM_City_State: '{result[1]}'")
        print(f"  MCC: {result[2]}")
        print(f"  Merchant_Name: {result[3]}")
        print()

Testing v3 parser:

Test 1:
  ATM_Address: 70 SPRINGER DR
  ATM_City_State: 'BANGORME'
  MCC: 5200
  Merchant_Name: LOWE S #1940



Using index counts, able to identify the MCC code correctly but the name loses its original value

In [11]:
def parse_line_v4(line):
    """Use field count to determine MCC position"""
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts  # clean line - no parsing needed
    
    # Right 10 columns are stable
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # Determine structure by field count
    if len(left_remainder) == 4:
        # 14 total fields: ATM_Address, ATM_City_State, MCC, Merchant_Name
        atm_address = left_remainder[0]
        atm_city_state = left_remainder[1]
        merchant_category = left_remainder[2]
        merchant_name = left_remainder[3]
        
    elif len(left_remainder) == 5:
        # 15 total fields: ATM_Address, Merchant_Street, Merchant_City, MCC, Merchant_Name
        atm_address = left_remainder[0]
        atm_city_state = ''  # NULL for card transactions
        merchant_category = left_remainder[3]
        # Combine merchant location parts with name
        merchant_name = f"{left_remainder[1]},{left_remainder[2]},{left_remainder[4]}"
        
    elif len(left_remainder) == 6:
        # 16 total fields: likely extra comma in merchant name
        atm_address = left_remainder[0]
        atm_city_state = ''
        merchant_category = left_remainder[3]
        merchant_name = ','.join(left_remainder[1:3] + left_remainder[4:])
        
    else:
        return None  # Unexpected structure
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Test again
print("Testing v4 parser:\n")
for i, line in enumerate(test_lines, 1):
    result = parse_line_v4(line)
    if result:
        print(f"Test {i}:")
        print(f"  ATM_Address: {result[0]}")
        print(f"  ATM_City_State: '{result[1]}'")
        print(f"  MCC: {result[2]}")
        print(f"  Merchant_Name: {result[3]}")
        print()
    else:
        print(f"Test {i}: FAILED TO PARSE")
        print()

Testing v4 parser:

Test 1:
  ATM_Address: 70 SPRINGER DR
  ATM_City_State: 'BANGORME'
  MCC: 5200
  Merchant_Name: LOWE S #1940

Test 2:
  ATM_Address: 7124
  ATM_City_State: ''
  MCC: 5815
  Merchant_Name: Covered Bridge,AUSTINTX,TRUST ME MOVIE

Test 3:
  ATM_Address: 1178
  ATM_City_State: ''
  MCC: 5712
  Merchant_Name:  701 Tillery S,AUSTINTX,WWW.AETICON.COM



In [12]:
def parse_line_v5(line):
    """More flexible parser for complex addresses"""
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts  # clean line
    
    # Right 10 columns are stable
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # Find MCC: look for 3-4 digit number in positions 2-5
    # MCCs are typically 4 digits starting with 0-7
    mcc_index = None
    for i in range(2, min(7, len(left_remainder))):  # Check positions 2-6
        val = left_remainder[i].strip()
        if (val.isdigit() and 
            len(val) in [3, 4] and 
            (len(val) == 3 or val[0] in '01234567')):  # Valid MCC range
            mcc_index = i
            break
    
    if mcc_index is None:
        return None  # Can't find MCC
    
    # Build fields based on MCC position
    atm_address = left_remainder[0]
    
    if mcc_index == 2:
        # Standard: Address, City, MCC, Merchant
        atm_city_state = left_remainder[1]
        merchant_category = left_remainder[2]
        merchant_name = ','.join(left_remainder[3:])
    else:
        # Complex address: Address, [location parts], MCC, Merchant
        atm_city_state = ''  # NULL - card transaction with merchant location
        merchant_category = left_remainder[mcc_index]
        # Everything between address and MCC is merchant location
        # Everything after MCC is merchant name
        merchant_location = ','.join(left_remainder[1:mcc_index])
        merchant_name_part = ','.join(left_remainder[mcc_index+1:])
        merchant_name = f"{merchant_location},{merchant_name_part}" if merchant_location else merchant_name_part
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Test on failed samples
failed_test_lines = [
    "Wine, Beer, and Spi,GRAND ISLANDNE,5921,Wine, Beer, and,20,POS Debit        Pri,P,Pinned Settlements  ,86,43.56,2025-12-08,182440, ,BSB169391",
    "A,A,RIO SEGUNDO.BO,,ALAJUELA,5331,TORO BURGUER ATO,40,D/C SETTLEMENT,X,Pinless Settlements ,00,40.29,2025-11-21,120855, ,BSB473858",
    "A,A,A,BO,CTRO.DENTR,ALAJUELA,5814,CAFE Y BAKERY P2,40,D/C SETTLEMENT,X,Pinless Settlements ,25,20.03,2025-11-30,102303, ,BSB246537"
]

print("Testing v5 on failed samples:\n")
for i, line in enumerate(failed_test_lines, 1):
    result = parse_line_v5(line)
    if result:
        print(f"Test {i}: ")
        print(f"  MCC: {result[2]}")
        print(f"  Merchant: {result[3][:60]}...")
        print()
    else:
        print(f"Test {i}: Still fails")
        print()

Testing v5 on failed samples:

Test 1: 
  MCC: 5921
  Merchant:  Beer, and Spi,GRAND ISLANDNE,Wine, Beer, and...

Test 2: 
  MCC: 5331
  Merchant: A,RIO SEGUNDO.BO,,ALAJUELA,TORO BURGUER ATO...

Test 3: 
  MCC: 5814
  Merchant: A,A,BO,CTRO.DENTR,ALAJUELA,CAFE Y BAKERY P2...



Since the MCC code never starts at index 0, scanning from their did not make sense hence using indices 2 to 6 (only includes till 5)

In [ ]:
# Full cleaning with v5 parser
with open(raw_path, 'r', encoding='utf-8') as fin, \
     open(new_clean_path, 'w', encoding='utf-8', newline='') as fout:
    
    writer = csv.writer(fout)
    writer.writerow(columns)
    
    success, failed = 0, 0
    failed_samples = []
    
    for i, line in enumerate(fin):
        if not line.strip():
            continue
        
        parsed = parse_line_v5(line)
        if parsed:
            writer.writerow(parsed)
            success += 1
        else:
            failed += 1
            if len(failed_samples) < 10:
                failed_samples.append((i+1, line.strip()))

print(f"Cleaned: {success:,} rows")
print(f"Failed:  {failed:,} rows ({failed/success*100:.4f}%)")
print(f"Output: {new_clean_path}")

if failed_samples:
    print("\n Failed samples:")
    for line_num, sample in failed_samples:
        print(f"  Line {line_num}: {sample[:150]}...")
else:
    print("\n All rows parsed successfully!")

Cleaned: 8,226,905 rows
Failed:  7,187 rows (0.0874%)
Output: /home/omkar/ds5500/data/processed/transactions_clean_v2.csv

 Failed samples:
  Line 21439: 1-800 CONTACTS, INC,DRAPERUT,8043,1-800 CONTACTS,,20,POS Debit        Pri,P,Pinned Settlements  ,77,68.98,2025-11-22,120910, ,BSB211734...
  Line 54578: LIDSFOUNDATION.ORG,INDIANAPOLISIN,8398,LIDS FOUNDATION,,20,POS Debit        Pri,P,Pinned Settlements  ,19,.01,2025-11-26,120517, ,BSB314594...
  Line 87708: 10 MOULTON STREET,,PORTLANDME,8111,PORT CITY LEGAL,20,POS Debit        Pri,P,Pinned Settlements  ,55,2000.00,2025-12-04,51649, ,BSB514670...
  Line 88415: 1-800 CONTACTS, INC,DRAPERUT,8043,1-800 CONTACTS,,20,POS Debit        Pri,P,Pinned Settlements  ,75,203.28,2025-11-27,80647, ,BSB390121...
  Line 112761: 1-800 CONTACTS, INC,DRAPERUT,8043,1-800 CONTACTS,,20,POS Debit        Pri,P,Pinned Settlements  ,28,59.99,2025-12-01,65727, ,BSB470338...
  Line 117344: 1-800 CONTACTS, INC,DRAPERUT,8043,1-800 CONTACTS,,20,POS Debit        Pri,

The above parser failed as it could not identify MCC codes starting from digits 8 

In [19]:
def parse_line_v6(line):
    """Fixed: Allow all valid MCC ranges 0000-9999"""
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts  # clean line
    
    # Right 10 columns are stable
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # Find MCC: 3-4 digit number in positions 2-6
    mcc_index = None
    for i in range(2, min(7, len(left_remainder))):
        val = left_remainder[i].strip()
        if val.isdigit() and len(val) in [3, 4]:  # Any 3-4 digit number
            mcc_index = i
            break
    
    if mcc_index is None:
        return None
    
    # Build fields based on MCC position
    atm_address = left_remainder[0]
    
    if mcc_index == 2:
        # Standard: Address, City, MCC, Merchant
        atm_city_state = left_remainder[1]
        merchant_category = left_remainder[2]
        merchant_name = ','.join(left_remainder[3:])
    else:
        # Complex: Address, [location], MCC, Merchant
        atm_city_state = ''  # NULL for card transactions
        merchant_category = left_remainder[mcc_index]
        merchant_location = ','.join(left_remainder[1:mcc_index])
        merchant_name_part = ','.join(left_remainder[mcc_index+1:])
        merchant_name = f"{merchant_location},{merchant_name_part}" if merchant_location else merchant_name_part
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Test on new failed samples
new_failed = [
    "1-800 CONTACTS, INC,DRAPERUT,8043,1-800 CONTACTS,,20,POS Debit        Pri,P,Pinned Settlements  ,77,68.98,2025-11-22,120910, ,BSB211734",
    "LIDSFOUNDATION.ORG,INDIANAPOLISIN,8398,LIDS FOUNDATION,,20,POS Debit        Pri,P,Pinned Settlements  ,19,.01,2025-11-26,120517, ,BSB314594"
]

print("Testing v6 on 8xxx MCC samples:\n")
for i, line in enumerate(new_failed, 1):
    result = parse_line_v6(line)
    if result:
        print(f"Test {i}:")
        print(f"  MCC: {result[2]}")
        print(f"  Merchant: {result[3][:50]}")
        print()
    else:
        print(f"Test {i}:")

Testing v6 on 8xxx MCC samples:

Test 1:
  MCC: 8043
  Merchant:  INC,DRAPERUT,1-800 CONTACTS,

Test 2:
  MCC: 8398
  Merchant: LIDS FOUNDATION,



- Initial versions (v1 to v3) : After parsing from left to right looking for a 4-digit number, the python parser was failing to identify because of street numbers and leaving everything in the merchant name column
- v4 : It relied on hard coded positions where MCC would always be considered at the 3rd index. It identified the MCC but failed to distinguish the other columns correctly
- v5 : It looked for a 4 digit number starting from 0-7 and only looked for indices starting from 2 till 5
- v6 : Ignored the first two items (index 0 and index 1). So successfully eliminated the street numbers and looked for a 4 digit number starting from 0-8 after that.


In [20]:
# Full cleaning with v6 parser
with open(raw_path, 'r', encoding='utf-8') as fin, \
     open(new_clean_path, 'w', encoding='utf-8', newline='') as fout:
    
    writer = csv.writer(fout)
    writer.writerow(columns)
    
    success, failed = 0, 0
    failed_samples = []
    
    for i, line in enumerate(fin):
        if not line.strip():
            continue
        
        parsed = parse_line_v6(line)
        if parsed:
            writer.writerow(parsed)
            success += 1
        else:
            failed += 1
            if len(failed_samples) < 20:  # Get more samples this time
                failed_samples.append((i+1, line.strip()))

print(f"Cleaned: {success:,} rows")
print(f"Failed:  {failed:,} rows ({failed/success*100:.4f}% if failed > 0 else 0)")
print(f"Output: {new_clean_path}")

if failed_samples:
    print(f"\n Failed samples (showing {len(failed_samples)}):")
    for line_num, sample in failed_samples:
        parts = sample.split(',')
        print(f"  Line {line_num} ({len(parts)} parts): {sample[:120]}...")
else:
    print("\n All rows parsed successfully!")

Cleaned: 8,234,091 rows
Failed:  1 rows (0.0000% if failed > 0 else 0)
Output: /home/omkar/ds5500/data/processed/transactions_clean_v2.csv

 Failed samples (showing 1):
  Line 8234093 (1 parts): (8234091 rows affected)...


In [ ]:
import pandas as pd

# Load both versions
old_df = pd.read_csv('../data/processed/transactions_clean.csv')
new_df = pd.read_csv('../data/processed/transactions_clean_v2.csv')

print("=" * 80)
print("COMPARISON: Old vs New Parsing")
print("=" * 80)

print(f"\n Row counts:")
print(f"  Old: {len(old_df):,}")
print(f"  New: {len(new_df):,}")
print(f"  Difference: {len(new_df) - len(old_df):,} rows")

print(f"\n Merchant_Category (MCC) comparison:")
print(f"  Old unique MCCs: {old_df['Merchant_Category'].nunique()}")
print(f"  New unique MCCs: {new_df['Merchant_Category'].nunique()}")

print(f"\n Old top 10 MCCs:")
for mcc, count in old_df['Merchant_Category'].value_counts().head(10).items():
    print(f"    {mcc}: {count:,}")

print(f"\n New top 10 MCCs:")
for mcc, count in new_df['Merchant_Category'].value_counts().head(10).items():
    print(f"    {mcc}: {count:,}")

# Check for invalid MCCs in old data
print(f"\n Invalid MCC check (non-numeric or >4 digits):")
old_invalid = old_df[~old_df['Merchant_Category'].astype(str).str.match(r'^\d{3,4}$')]
new_invalid = new_df[~new_df['Merchant_Category'].astype(str).str.match(r'^\d{3,4}$')]
print(f"  Old invalid MCCs: {len(old_invalid):,} ({len(old_invalid)/len(old_df)*100:.2f}%)")
print(f"  New invalid MCCs: {len(new_invalid):,} ({len(new_invalid)/len(new_df)*100:.2f}%)")

if len(old_invalid) > 0:
    print(f"\n  Sample old invalid MCCs:")
    print(old_invalid['Merchant_Category'].value_counts().head(10))

print(f"\n ATM_City_State NULL/empty counts:")
old_null = (old_df['ATM_City_State'].isna() | (old_df['ATM_City_State']=='')).sum()
new_null = (new_df['ATM_City_State'].isna() | (new_df['ATM_City_State']=='')).sum()
print(f"  Old: {old_null:,} ({old_null/len(old_df)*100:.2f}%)")
print(f"  New: {new_null:,} ({new_null/len(new_df)*100:.2f}%)")

# Check our specific test case
print(f"\n BSB066832 transaction check:")
old_sample = old_df[old_df['CustomerID'] == 'BSB066832'].iloc[0]
new_sample = new_df[new_df['CustomerID'] == 'BSB066832'].iloc[0]

print(f"  Old:")
print(f"    MCC: {old_sample['Merchant_Category']}")
print(f"    Merchant: {old_sample['Merchant_Name'][:60]}")
print(f"  New:")
print(f"    MCC: {new_sample['Merchant_Category']}")
print(f"    Merchant: {new_sample['Merchant_Name'][:60]}")

COMPARISON: Old vs New Parsing

📊 Row counts:
  Old: 8,234,091
  New: 8,234,091
  Difference: 0 rows

  Merchant_Category (MCC) comparison:
  Old unique MCCs: 645
  New unique MCCs: 478

  Old top 10 MCCs:
    5411: 1,354,441
    5814: 972,379
    5541: 780,367
    5812: 490,537
    5542: 414,365
    5818: 293,027
    5310: 254,534
    4899: 236,487
    5942: 218,654
    5499: 150,422

  New top 10 MCCs:
    5411: 1,354,757
    5814: 972,412
    5541: 780,466
    5812: 490,573
    5542: 414,373
    5818: 293,027
    5310: 254,534
    4899: 236,487
    5942: 218,659
    5499: 150,502

 Invalid MCC check (non-numeric or >4 digits):
  Old invalid MCCs: 1 (0.00%)
  New invalid MCCs: 0 (0.00%)

  Sample old invalid MCCs:
Merchant_Category
1    1
Name: count, dtype: int64

 ATM_City_State NULL/empty counts:
  Old: 996 (0.01%)
  New: 91 (0.00%)

 BSB066832 transaction check:
  Old:
    MCC: 7333
    Merchant: CANVA* I04721-44
  New:
    MCC: 7333
    Merchant: CANVA* I04721-44


In [ ]:
# Filter for the SPECIFIC transaction that was broken
# (Customer + Date + Amount)
check_cust = 'BSB066832'
check_date = '2025-11-23'
check_amt = 49.00

print(f"Checking specific BROKEN transaction for {check_cust}:")

# Get the row from the NEW dataframe
broken_row_new = new_df[
    (new_df['CustomerID'] == check_cust) & 
    (new_df['Transaction_Date'] == check_date) & 
    (abs(new_df['Amount_Completed'] - check_amt) < 0.01) # Float safety
]

if not broken_row_new.empty:
    row = broken_row_new.iloc[0]
    print("\n NEW PARSER RESULT:")
    print(f"  Address:  {row['ATM_Address']}")
    print(f"  City:     '{row['ATM_City_State']}'")     # Should be empty or clean
    print(f"  MCC:      {row['Merchant_Category']}")    # Should be 5815
    print(f"  Name:     {row['Merchant_Name']}")        # Should contain 'Covered Bridge'
else:
    print("Could not find the specific transaction.")

Checking specific BROKEN transaction for BSB066832:

 NEW PARSER RESULT:
  Address:  7124,Covered Bridge
  City:     'AUSTINTX'
  MCC:      5815
  Name:     TRUST ME MOVIE


In [28]:
# Find the EXACT transaction we tested (date + amount match)
test_transaction_old = old_df[
    (old_df['CustomerID'] == 'BSB066832') & 
    (old_df['Transaction_Date'] == '11/23/2025') &
    (old_df['Amount_Completed'] == 49.0)
]

test_transaction_new = new_df[
    (new_df['CustomerID'] == 'BSB066832') & 
    (new_df['Transaction_Date'] == '11/23/2025') &
    (new_df['Amount_Completed'] == 49.0)
]

print("SPECIFIC TEST TRANSACTION (Date: 11/23/2025, Amount: $49):")
if len(test_transaction_old) > 0:
    print(f"\nOld:")
    print(f"  MCC: {test_transaction_old.iloc[0]['Merchant_Category']}")
    print(f"  Merchant: {test_transaction_old.iloc[0]['Merchant_Name']}")
    print(f"  City: {test_transaction_old.iloc[0]['ATM_City_State']}")
else:
    print("\nOld: NOT FOUND")

if len(test_transaction_new) > 0:
    print(f"\nNew:")
    print(f"  MCC: {test_transaction_new.iloc[0]['Merchant_Category']}")
    print(f"  Merchant: {test_transaction_new.iloc[0]['Merchant_Name']}")
    print(f"  City: '{test_transaction_new.iloc[0]['ATM_City_State']}'")
else:
    print("\nNew: NOT FOUND")

# Also check how many transactions this customer has
print(f"\n\nTotal transactions for BSB066832:")
print(f"  Old: {len(old_df[old_df['CustomerID'] == 'BSB066832'])}")
print(f"  New: {len(new_df[new_df['CustomerID'] == 'BSB066832'])}")

SPECIFIC TEST TRANSACTION (Date: 11/23/2025, Amount: $49):

Old: NOT FOUND

New: NOT FOUND


Total transactions for BSB066832:
  Old: 94
  New: 94


In [29]:
# Check all BSB066832 transactions to find our test case
print("All BSB066832 transactions:")
bsb066832_old = old_df[old_df['CustomerID'] == 'BSB066832'][
    ['Transaction_Date', 'Amount_Completed', 'Merchant_Category', 'Merchant_Name']
].head(20)
print("\nOld data:")
print(bsb066832_old)

bsb066832_new = new_df[new_df['CustomerID'] == 'BSB066832'][
    ['Transaction_Date', 'Amount_Completed', 'Merchant_Category', 'Merchant_Name']
].head(20)
print("\nNew data:")
print(bsb066832_new)

# Look for the $49 transaction specifically
print("\n" + "="*80)
print("Looking for $49 transaction:")
old_49 = old_df[(old_df['CustomerID'] == 'BSB066832') & (old_df['Amount_Completed'] == 49.0)]
new_49 = new_df[(new_df['CustomerID'] == 'BSB066832') & (new_df['Amount_Completed'] == 49.0)]

if len(old_49) > 0:
    print("\nOld $49 transaction:")
    print(f"  Date: {old_49.iloc[0]['Transaction_Date']}")
    print(f"  MCC: {old_49.iloc[0]['Merchant_Category']}")
    print(f"  Merchant: {old_49.iloc[0]['Merchant_Name']}")
    print(f"  City: '{old_49.iloc[0]['ATM_City_State']}'")

if len(new_49) > 0:
    print("\nNew $49 transaction:")
    print(f"  Date: {new_49.iloc[0]['Transaction_Date']}")
    print(f"  MCC: {new_49.iloc[0]['Merchant_Category']}")
    print(f"  Merchant: {new_49.iloc[0]['Merchant_Name']}")
    print(f"  City: '{new_49.iloc[0]['ATM_City_State']}'")

All BSB066832 transactions:

Old data:
        Transaction_Date  Amount_Completed  Merchant_Category  \
822733        2025-12-07             60.00               7333   
855143        2025-11-26             22.19               4816   
1283647       2025-12-11             13.99               5815   
1494863       2025-12-22             15.33               5814   
2287111       2025-12-28             37.18               4816   
2293571       2025-12-28             37.18               4816   
2428540       2026-01-06             30.00               5814   
2946025       2025-11-23             49.00               7124   
2947744       2025-11-22             10.46               5734   
2963010       2025-11-22             10.46               5734   
2979701       2025-11-22             11.48               7372   
3016850       2025-11-19             71.75               5734   
3022276       2025-11-22              6.00               5499   
3045331       2025-11-22             45.00         

In [ ]:
def parse_line_v7(line):
    """
    Precision Parser: Uses MCC position to correctly identify City vs Address.
    Assumption: The field immediately BEFORE the MCC is always the City/State.
    """
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts 
    
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    # 1. FIND THE ANCHOR (Same robust logic as v6)
    mcc_index = None
    # We scan indices 2 to 6 to find the MCC
    for i in range(2, min(7, len(left_remainder))):
        val = left_remainder[i].strip()
        if val.isdigit() and len(val) in [3, 4]:
            mcc_index = i
            break
            
    if mcc_index is None:
        return None

    # 2. EXTRACT DATA RELATIVE TO THE ANCHOR
    
    # MCC is at the anchor index
    merchant_category = left_remainder[mcc_index].strip()
    
    # MERCHANT NAME is everything AFTER the anchor
    merchant_name = ','.join(left_remainder[mcc_index+1:])
    
    # CITY/STATE is the neighbor immediately LEFT of the anchor
    # (Since we search from index 2, index-1 is always safe)
    atm_city_state = left_remainder[mcc_index - 1].strip()
    
    # ADDRESS is everything LEFT of the City
    # We join it all back together (fixing the split street numbers)
    atm_address = ','.join(left_remainder[:mcc_index - 1])
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Testing on specific broken row
broken_row = "7124,Covered Bridge,AUSTINTX,5815,TRUST ME MOVIE,40,D/C SETTLEMENT,X,Pinless Settlements ,24,49.00,2025-11-23,91146, ,BSB066832"
result = parse_line_v7(broken_row)

print("v7 PRECISION RESULT")
print(f"Address:  '{result[0]}'")  
print(f"City:     '{result[1]}'")
print(f"MCC:      '{result[2]}'")
print(f"Name:     '{result[3]}'")

--- v7 PRECISION RESULT ---
Address:  '7124,Covered Bridge'
City:     'AUSTINTX'
MCC:      '5815'
Name:     'TRUST ME MOVIE'


In [25]:
# Full cleaning with v7 parser
with open(raw_path, 'r', encoding='utf-8') as fin, \
     open(new_clean_path, 'w', encoding='utf-8', newline='') as fout:
    
    writer = csv.writer(fout)
    writer.writerow(columns)
    
    success, failed = 0, 0
    failed_samples = []
    
    for i, line in enumerate(fin):
        if not line.strip():
            continue
        
        parsed = parse_line_v7(line)
        if parsed:
            writer.writerow(parsed)
            success += 1
        else:
            failed += 1
            if len(failed_samples) < 20:  # Get more samples this time
                failed_samples.append((i+1, line.strip()))

print(f"Cleaned: {success:,} rows")
print(f"Failed:  {failed:,} rows ({failed/success*100:.4f}% if failed > 0 else 0)")
print(f"Output: {new_clean_path}")

if failed_samples:
    print(f"\n Failed samples (showing {len(failed_samples)}):")
    for line_num, sample in failed_samples:
        parts = sample.split(',')
        print(f"  Line {line_num} ({len(parts)} parts): {sample[:120]}...")
else:
    print("\n All rows parsed successfully!")

Cleaned: 8,234,091 rows
Failed:  1 rows (0.0000% if failed > 0 else 0)
Output: /home/omkar/ds5500/data/processed/transactions_clean_v2.csv

 Failed samples (showing 1):
  Line 8234093 (1 parts): (8234091 rows affected)...


In [ ]:
# Filter for rows where City is still NULL/Empty in the NEW data
blank_city_rows = new_df[
    (new_df['ATM_City_State'].isna()) | 
    (new_df['ATM_City_State'] == '') | 
    (new_df['ATM_City_State'] == 'nan')
]

print(f"Inspecting {len(blank_city_rows)} rows with blank cities:\n")

# Show the columns involved in the split
print(blank_city_rows[['ATM_Address', 'ATM_City_State', 'Merchant_Category', 'Merchant_Name', 'CustomerID']].head(10))

# To see the specific Customer IDs to find them in the raw file
print("\nSample Customer IDs to check in raw file:")
print(blank_city_rows['CustomerID'].head(5).tolist())

Inspecting 91 rows with blank cities:

                            ATM_Address ATM_City_State  Merchant_Category  \
3475496             90 WOODLANDS INDUST            NaN               5817   
3585609                      Hurst Lane            NaN               8911   
3655202  Room B3, 19/F, Tung,SHEUNG WAN            NaN               5691   
3665063    HOLLY HOUSE, HIGH R,GRANTHAM            NaN               5734   
3673738             90 WOODLANDS INDUST            NaN               5817   
3743021      17-19, St. Georges,NORWICH            NaN               5945   
3825248             90 WOODLANDS INDUST            NaN               5817   
3897141      3rd Floor Waverly H,LONDON            NaN               7929   
3904058            7 The Close,,NORWICH            NaN               5818   
4079855             90 WOODLANDS INDUST            NaN               5817   

            Merchant_Name CustomerID  
3475496          Netshort  BSB192979  
3585609               NaN  BSB37267

In [ ]:
# The IDs from sample output
target_ids = ['BSB192979', 'BSB372676', 'BSB005675', 'BSB073391', 'BSB269534']

print(f"Searching raw file for {len(target_ids)} specific customers...\n")

with open('../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv', 'r') as f:
    for i, line in enumerate(f, 1):
        # Check if any of our target IDs are in this line
        for target in target_ids:
            if target in line:
                print(f"--- Found {target} at Line {i} ---")
                print(f"Raw: {line.strip()}")
                
                # Show how v7 would see the 'left remainder'
                parts = line.strip().split(',')
                left_remainder = parts[:-10]
                print(f"Left Side Parts: {left_remainder}")
                print("-" * 50 + "\n")

Searching raw file for 5 specific customers...

--- Found BSB073391 at Line 17938 ---
Raw: HANNAFORD #8143 180,BOOTHBAY HBRME,5411,HANNAFORD #8143,20,POS Debit        Pri,P,Pinned Settlements  ,39,27.45,2025-11-25,183107, ,BSB073391
Left Side Parts: ['HANNAFORD #8143 180', 'BOOTHBAY HBRME', '5411', 'HANNAFORD #8143']
--------------------------------------------------

--- Found BSB269534 at Line 25863 ---
Raw: 900 STILLWATER AVE,BANGORME,5411,WM SUPERCENTER #,20,POS Debit        Pri,P,Pinned Settlements  ,29,57.19,2025-11-26,155530, ,BSB269534
Left Side Parts: ['900 STILLWATER AVE', 'BANGORME', '5411', 'WM SUPERCENTER #']
--------------------------------------------------

--- Found BSB005675 at Line 49729 ---
Raw: 435 KENNEDY MEMORIA,WATERVILLEME,5814,MCDONALD S F2448,20,POS Debit        Pri,P,Pinned Settlements  ,00,10.00,2025-12-05,140700, ,BSB005675
Left Side Parts: ['435 KENNEDY MEMORIA', 'WATERVILLEME', '5814', 'MCDONALD S F2448']
-------------------------------------------------

In [ ]:
# v7 Parser ---
def parse_line_v7(line):
    line = line.strip()
    parts = line.split(',')
    
    if len(parts) == 14:
        return parts 
    
    right_10 = parts[-10:]
    left_remainder = parts[:-10]
    
    mcc_index = None
    # Scan indices 2 to 6 to find the MCC
    for i in range(2, min(7, len(left_remainder))):
        val = left_remainder[i].strip()
        if val.isdigit() and len(val) in [3, 4]:
            mcc_index = i
            break
            
    if mcc_index is None:
        return None

    merchant_category = left_remainder[mcc_index].strip()
    merchant_name = ','.join(left_remainder[mcc_index+1:])
    
    # CITY is the neighbor to the LEFT of MCC
    atm_city_state = left_remainder[mcc_index - 1].strip()
    
    # ADDRESS is everything LEFT of the City
    atm_address = ','.join(left_remainder[:mcc_index - 1])
    
    return [atm_address, atm_city_state, merchant_category, merchant_name] + right_10

# Scanner 
raw_path = '../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv'

print(f"Scanning {raw_path} for blank cities...")
print("="*80)

blank_count = 0

with open(raw_path, 'r') as f:
    for i, line in enumerate(f):
        if not line.strip(): continue
        
        parsed = parse_line_v7(line)
        
        if parsed:
            # parsed[1] is the City column
            city = parsed[1]
            
            # Check for Empty String OR "NA" (which Pandas treats as Null)
            if city == '' or city == 'NA':
                blank_count += 1
                
                # Print the "Smoking Gun" evidence
                print(f"Line {i+1}:")
                print(f" RAW:    {line.strip()}")
                print(f" PARSED: Addr='{parsed[0]}' | City='{city}' | MCC='{parsed[2]}'")
                print("-" * 50)

print(f"\nTotal rows with blank/NA cities found in raw file: {blank_count}")


Scanning ../data/raw/OmkarCapstone_Final_ArchivePassportData_20260123.csv for blank cities...
Line 3475497:
 RAW:    90 WOODLANDS INDUST,NA,5817,Netshort,40,D/C SETTLEMENT,X,Pinless Settlements ,95,20.00,2025-11-27,231044, ,BSB192979
 PARSED: Addr='90 WOODLANDS INDUST' | City='NA' | MCC='5817'
--------------------------------------------------
Line 3585610:
 RAW:    Hurst Lane,NA,8911,NA,40,D/C SETTLEMENT,X,Pinless Settlements ,20,141.54,2025-11-24,181144, ,BSB372676
 PARSED: Addr='Hurst Lane' | City='NA' | MCC='8911'
--------------------------------------------------
Line 3655203:
 RAW:    Room B3, 19/F, Tung,SHEUNG WAN,,5691,SP ATTIFIT,40,D/C SETTLEMENT,X,Pinless Settlements ,66,26.08,2025-11-28,71453, ,BSB005675
 PARSED: Addr='Room B3, 19/F, Tung,SHEUNG WAN' | City='' | MCC='5691'
--------------------------------------------------
Line 3665064:
 RAW:    HOLLY HOUSE, HIGH R,GRANTHAM,,5734,SP CONTOUR FABRI,40,D/C SETTLEMENT,X,Pinless Settlements ,37,98.00,2025-11-22,143552, ,BSB073391